# 04. Comprehensive Analysis

Compares the hierarchical (multi-stage) classification approach against a
single flat 18-class classifier, using the `all_predictions.xlsx` exports
written by the training notebooks. This is where the headline comparison in
the README (flat vs. hierarchical, Table in section 4) comes from.

Sections (see `src/sake_rice_inspection/analysis.py`):
1. Per-layer accuracy, and the effect of the multi-label domain-logic correction.
2. Hierarchical vs. flat classification, end to end.
3. A focused look at Shinpaku detection specifically (the single most
   important trait for sake brewing suitability).
4. Multi-label (Shinpaku/Base White/Back White/Belly White) comparison.
5. Cross-architecture comparison for a given training run.

> **Note on data**: this notebook expects `all_predictions.xlsx` files
> produced by `03_Training_MultiLabel.ipynb` (and its counterparts for the
> other classification heads). Since those depend on the NDA-protected
> dataset, no prediction files are included in this repository — the paths
> below use glob patterns so you only need to point `RESULTS_ROOT` at your
> own `outputs/` directory.

In [ ]:
import sys
import glob
from pathlib import Path

sys.path.insert(0, str(Path("..") / "src"))

import numpy as np
import pandas as pd

from sake_rice_inspection.analysis import (
    analyze_feature_binary,
    apply_pw_correction,
    collect_binary_vectors,
    compute_classification_metrics,
    format_metrics_markdown,
    hamming_accuracy_label_mean,
    integrate_hierarchical_predictions,
    labels_to_presence_strings,
    plot_confusion_matrix,
)

## 1. Load the latest predictions for each classification head

Each training run writes to its own timestamped experiment directory (see
`experiment.create_experiment_dir`); `latest_predictions` picks the most
recently modified `all_predictions.xlsx` for a given head/architecture.

In [ ]:
RESULTS_ROOT = Path("../outputs")
OUT_DIR = Path("../outputs/classfi_ana")
OUT_DIR.mkdir(parents=True, exist_ok=True)

ALLOWED_PW_LABELS = [
    "心白", "背白", "腹白", "基白",
    "心白+背白", "心白+腹白", "心白+基白",
    "背白+腹白", "基白+腹白", "心白+背白+腹白",
]
CLASS_DISPLAY_ORDER = [
    "砕粒", "発芽", "病害", "斑点", "青未熟",
    "基白+腹白", "基白", "心白+基白",
    "背白", "背白+腹白", "心白+背白",
    "腹白", "心白", "心白+背白+腹白", "心白+腹白",
    "死米", "乳白", "整粒",
]


def latest_predictions(pattern: str) -> str:
    """Returns the most recently modified match for a glob pattern, or None."""
    matches = glob.glob(str(RESULTS_ROOT / pattern))
    return max(matches, key=lambda p: Path(p).stat().st_mtime) if matches else None


layer_paths = {
    "layer1": latest_predictions("train_grouped/*/*/all_predictions.xlsx"),
    "layer2_chakusyoku": latest_predictions("train_chakusyoku/*/*/all_predictions.xlsx"),
    "layer2_opaque": latest_predictions("train_opaque/*/*/all_predictions.xlsx"),
    "layer2_part_white": latest_predictions("train_part_white/*/*/all_predictions.xlsx"),
    "all_in_one": latest_predictions("train_all/*/*/all_predictions.xlsx"),
}
missing = [k for k, v in layer_paths.items() if v is None]
if missing:
    print(f"No predictions found yet for: {missing}. Run 03_Training_MultiLabel.ipynb (and its siblings) first.")

dfs = {k: pd.read_excel(v) for k, v in layer_paths.items() if v is not None}

## 2. Per-layer accuracy, with and without domain-logic correction

In [ ]:
md_lines = ["# Classification accuracy summary"]

for key, title in [("layer1", "Layer 1 (coarse)"), ("layer2_chakusyoku", "Layer 2 (colored)"), ("layer2_opaque", "Layer 2 (opaque white)")]:
    if key not in dfs:
        continue
    df = dfs[key]
    metrics = compute_classification_metrics(df.iloc[:, 2], df.iloc[:, 3])
    plot_confusion_matrix(df.iloc[:, 2], df.iloc[:, 3], title=title, save_path=str(OUT_DIR / f"confusion_matrix_{key}.png"), desired_order=CLASS_DISPLAY_ORDER)
    md_lines += format_metrics_markdown(title, metrics)

if "layer2_part_white" in dfs:
    df_pw = apply_pw_correction(dfs["layer2_part_white"], ALLOWED_PW_LABELS)
    dfs["layer2_part_white"] = df_pw  # keep the corrected column for later sections

    metrics_before = compute_classification_metrics(df_pw.iloc[:, 2], df_pw.iloc[:, 3])
    plot_confusion_matrix(df_pw.iloc[:, 2], df_pw.iloc[:, 3], title="Layer 2 (partially clouded, before correction)", save_path=str(OUT_DIR / "confusion_matrix_pw_before.png"), desired_order=CLASS_DISPLAY_ORDER)
    md_lines += format_metrics_markdown("Layer 2 (partially clouded, before correction)", metrics_before)

    metrics_after = compute_classification_metrics(df_pw.iloc[:, 2], df_pw["pred_corrected"], labels=ALLOWED_PW_LABELS)
    plot_confusion_matrix(df_pw.iloc[:, 2], df_pw["pred_corrected"], labels=ALLOWED_PW_LABELS, title="Layer 2 (partially clouded, after correction)", save_path=str(OUT_DIR / "confusion_matrix_pw_after.png"))
    md_lines += format_metrics_markdown("Layer 2 (partially clouded, after correction)", metrics_after)

## 3. Hierarchical vs. flat classification, end to end

A grain's final hierarchical prediction is its layer-2 prediction whenever
layer 1 routed it correctly; otherwise it falls back to the layer-1 label
(see `integrate_hierarchical_predictions`).

In [ ]:
hierarchical_computed = False
if all(k in dfs for k in ("layer1", "layer2_chakusyoku", "layer2_opaque", "layer2_part_white", "all_in_one")):
    df_l1, df_bulk = dfs["layer1"], dfs["all_in_one"]
    true_map = {str(r.iloc[1]): r.iloc[2] for _, r in df_bulk.iterrows()}

    layer2_all = pd.concat([dfs["layer2_chakusyoku"], dfs["layer2_opaque"], dfs["layer2_part_white"]])
    l2_map_raw = {str(r.iloc[1]): r.iloc[3] for _, r in layer2_all.iterrows()}
    l2_map_corrected = l2_map_raw.copy()
    l2_map_corrected.update({str(r.iloc[1]): r["pred_corrected"] for _, r in dfs["layer2_part_white"].iterrows()})

    df_overall_raw = integrate_hierarchical_predictions(df_l1, true_map, l2_map_raw)
    df_overall_corrected = integrate_hierarchical_predictions(df_l1, true_map, l2_map_corrected)
    df_bulk_renamed = pd.DataFrame({"true": df_bulk.iloc[:, 2], "pred": df_bulk.iloc[:, 3]})
    hierarchical_computed = True

    for label, df_result in [
        ("Hierarchical (raw)", df_overall_raw),
        ("Hierarchical (corrected)", df_overall_corrected),
        ("Flat (single classifier)", df_bulk_renamed),
    ]:
        metrics = compute_classification_metrics(df_result["true"], df_result["pred"])
        plot_confusion_matrix(df_result["true"], df_result["pred"], title=label, save_path=str(OUT_DIR / f"confusion_matrix_{label}.png"), desired_order=CLASS_DISPLAY_ORDER)
        md_lines += format_metrics_markdown(label, metrics)

    (OUT_DIR / "classfi_ana.md").write_text("\n".join(md_lines) + "\n", encoding="utf-8")

## 4. Focused analysis: Shinpaku detection

Overall macro-F1 can decline even as detection of the single most important
trait for sake brewing suitability (Shinpaku, the starch-white core)
improves — so it is evaluated as its own binary problem.

In [ ]:
if hierarchical_computed:
    for label, df_result in [("Flat classification", df_bulk_renamed), ("Hierarchical (corrected)", df_overall_corrected)]:
        y_true, y_pred = analyze_feature_binary(df_result, "心白")
        y_true_str = labels_to_presence_strings(y_true)
        y_pred_str = labels_to_presence_strings(y_pred)
        metrics = compute_classification_metrics(y_true_str, y_pred_str, labels=["無", "有"])
        title = f"Shinpaku detection ({label})"
        plot_confusion_matrix(y_true_str, y_pred_str, labels=["無", "有"], title=title, save_path=str(OUT_DIR / f"confusion_matrix_shinpaku_{label}.png"))
        print(title, metrics)

## 5. Multi-label comparison (Shinpaku / Base White / Back White / Belly White)

Rather than reducing everything to a single joined label, this evaluates
each of the four co-occurring traits as its own presence/absence call.

In [ ]:
TARGET_LABELS = ["心白", "背白", "腹白", "基白"]

if "layer2_part_white" in dfs:
    df_pw = dfs["layer2_part_white"]
    y_true_pw, y_pred_pw_raw = collect_binary_vectors(df_pw, TARGET_LABELS, pred_col=df_pw.columns[3], true_col=df_pw.columns[2])
    y_pred_pw_corrected = collect_binary_vectors(df_pw, TARGET_LABELS, pred_col="pred_corrected", true_col=df_pw.columns[2])[1]

    hamming_raw, per_label_raw = hamming_accuracy_label_mean(y_true_pw, y_pred_pw_raw)
    hamming_corrected, per_label_corrected = hamming_accuracy_label_mean(y_true_pw, y_pred_pw_corrected)

    print(f"Hamming accuracy (label mean), before correction: {hamming_raw:.4f}")
    print(f"Hamming accuracy (label mean), after correction:  {hamming_corrected:.4f}")

    for i, label in enumerate(TARGET_LABELS):
        print(f"  {label}: before={per_label_raw[i]:.4f}  after={per_label_corrected[i]:.4f}")
        plot_confusion_matrix(
            labels_to_presence_strings(y_true_pw[:, i]), labels_to_presence_strings(y_pred_pw_corrected[:, i]),
            labels=["無", "有"], title=f"{label} (corrected)",
            save_path=str(OUT_DIR / f"confusion_matrix_multilabel_{label}.png"),
        )

## 6. Cross-architecture comparison

Compares every architecture trained for a given classification head (e.g.
every `*_analysis` subfolder under `train_part_white`), picking each
architecture's most recent run.

In [ ]:
def compare_architectures(run_name: str, architectures: list, apply_correction: bool = False):
    out_dir = OUT_DIR / run_name
    out_dir.mkdir(parents=True, exist_ok=True)
    md = [f"# {run_name} architecture comparison"]

    for arch in architectures:
        path = latest_predictions(f"{run_name}/{arch}_analysis/*/all_predictions.xlsx")
        if path is None:
            print(f"No results yet for {run_name}/{arch}")
            continue

        df = pd.read_excel(path)
        if apply_correction:
            df = apply_pw_correction(df, ALLOWED_PW_LABELS)
            metrics = compute_classification_metrics(df.iloc[:, 2], df["pred_corrected"], labels=ALLOWED_PW_LABELS)
            plot_confusion_matrix(df.iloc[:, 2], df["pred_corrected"], labels=ALLOWED_PW_LABELS, title=f"{run_name}_{arch}", save_path=str(out_dir / f"confusion_matrix_{arch}.png"))
        else:
            metrics = compute_classification_metrics(df.iloc[:, 2], df.iloc[:, 3])
            plot_confusion_matrix(df.iloc[:, 2], df.iloc[:, 3], title=f"{run_name}_{arch}", save_path=str(out_dir / f"confusion_matrix_{arch}.png"), desired_order=CLASS_DISPLAY_ORDER)
        md += format_metrics_markdown(arch, metrics)

    (out_dir / f"classfi_ana_{run_name}.md").write_text("\n".join(md) + "\n", encoding="utf-8")


compare_architectures("train_all", ["resnet34", "resnet50", "efficientnet_b2"])
compare_architectures("train_grouped", ["resnet34", "resnet50", "efficientnet_b2", "efficientnet_b4"])
compare_architectures("train_chakusyoku", ["resnet34", "resnet50", "efficientnet_b2"])
compare_architectures("train_opaque", ["resnet34", "resnet50", "resnet101", "efficientnet_b2", "efficientnet_b4"])
compare_architectures("train_part_white", ["resnet34", "resnet50", "resnet101", "efficientnet_b2", "efficientnet_b4", "vit_b_16"], apply_correction=True)